# DentalX — Unified Evaluation

Aggregates test-set metrics across all three models into one view, mirroring the senior project's `testing/evaluate.ipynb`. Each model keeps its own weights in `models/` and its own `data.yaml`.

In [ ]:
pip -q install -U ultralytics
from ultralytics import YOLO
import json, os

MODELS = {
  "model1_occlusal":   {"w": "../model1_occlusal/models/occlusal_caries_det.pt", "data": "../model1_occlusal/data/data.yaml",   "imgsz":640},
  "model2_tooth_type": {"w": "../model2_tooth_type/models/best.pt",              "data": "../model2_tooth_type/data/data.yaml", "imgsz":640},
  "model3_gingival":   {"w": "../model3_gingival/models/best.pt",                "data": "../model3_gingival/data/data.yaml",   "imgsz":640},
}

## Run each model's test-split validation

In [ ]:
summary = {}
for name, cfg in MODELS.items():
    if not (os.path.exists(cfg['w']) and os.path.exists(cfg['data'])):
        print(f'skip {name}: missing weights or data.yaml'); continue
    m = YOLO(cfg['w'])
    mt = m.val(data=cfg['data'], split='test', imgsz=cfg['imgsz'], device=0, verbose=False)
    summary[name] = {'mAP50': round(float(mt.box.map50),4), 'mAP50_95': round(float(mt.box.map),4)}
    print(f"{name:20s} mAP50={summary[name]['mAP50']}  mAP50-95={summary[name]['mAP50_95']}")
json.dump(summary, open('unified_metrics.json','w'), indent=2)
print('\nwrote unified_metrics.json')